# Transformer Scaling Benchmark

Mede tempo de treino, predição e memória (RAM + VRAM) para os 6 modelos
transformer em função de N (500 → 50 000).

**Modelos:** FTTransformer × 4 atenções, SAINTColnorm, FTTransformerCURColnorm  
**N:** [500, 1000, 2000, 5000, 10 000, 20 000, 50 000]  
**Repetições:** 3 (mediana)

**Saída:** `results/transformer_scaling.json`

**Antes de rodar:** Settings → Accelerator → GPU T4 x2 (ou P100).

**Resume:** se a sessão cair, faça download do JSON em Output, suba como
dataset Kaggle (Add Data) e ajuste `RESUME_PATH` na Célula 5.

In [ ]:
# ── Célula 1: Verifica GPU ──────────────────────────────────────────────────
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memória: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('GPU não detectada — ative em Settings > Accelerator')

In [ ]:
# ── Célula 2: Clonar repo ───────────────────────────────────────────────────
import os, subprocess

GIT_URL     = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
PROJECT_DIR = '/kaggle/working/sparse-lssvm-transformers-study'

if os.path.exists(PROJECT_DIR):
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase'], check=True)
else:
    subprocess.run(['git', 'clone', GIT_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
!git log --oneline -3
print(f'Dir: {os.getcwd()}')

In [ ]:
# ── Célula 3: Dependências ──────────────────────────────────────────────────
!pip install -q entmax einops psutil
import torch, sklearn, numpy, psutil
print(f'torch {torch.__version__} | sklearn {sklearn.__version__} | numpy {numpy.__version__} | psutil {psutil.__version__}')

In [ ]:
# ── Célula 4: Estimativa de tempo ───────────────────────────────────────────
# FTTransformer (softmax/topk/entmax/sparsemax): atenção é O(p²), p=20 features
#   → escala linearmente com N, deve chegar até N=50K sem OOM
#   → estimativa: ~15-30 min por variante × 4 = 1-2h
# SAINTColnorm: atenção inter-instância O(N²)
#   → OOM esperado ~N=5K–10K (T4 16GB)
#   → estimativa: ~20 min antes do OOM
# FTTransformerCURColnorm: O(N·m), m=0.2N
#   → OOM esperado ~N=20K–50K (C = [N, m] fica ~7.5GB em N=50K)
#   → estimativa: ~40 min antes do OOM
# ─────────────────────────────────────────────────────────────────────────────
# Total estimado: 2–3 horas  (Kaggle T4, limite de sessão 12h)
print('Estimativa: 2–3 horas em T4.')
print('SAINT deve dar OOM em torno de N=5K–10K.')
print('FT-CUR deve dar OOM em torno de N=20K–50K.')
print('FTTransformer × 4 devem chegar até N=50K (atenção sobre p=20 features).')

In [ ]:
# ── Célula 5: Resume (opcional) ─────────────────────────────────────────────
import shutil, json
from pathlib import Path

OUT = Path('results/transformer_scaling.json')
OUT.parent.mkdir(exist_ok=True)

# Se quiser retomar de onde parou:
# 1. Suba o JSON anterior como Kaggle dataset (Add Data)
# 2. Ajuste o caminho abaixo e descomente

# RESUME_PATH = Path('/kaggle/input/SEU-DATASET/transformer_scaling.json')
# if RESUME_PATH.exists():
#     shutil.copy(RESUME_PATH, OUT)
#     records = json.loads(OUT.read_text())
#     print(f'Restaurado: {len(records)} entradas')

if OUT.exists():
    n = len(json.loads(OUT.read_text()))
    print(f'transformer_scaling.json: {n} entradas já presentes')
else:
    print('transformer_scaling.json: começando do zero')

In [ ]:
# ── Célula 6: Rodar benchmark ───────────────────────────────────────────────
# Copia resultado para /kaggle/working/ a cada modelo (caso a sessão caia)
!python -u scripts/run_transformer_scaling.py \
    --output results/transformer_scaling.json \
    --repeats 3 \
    2>&1 | tee /kaggle/working/scaling.log

import shutil
shutil.copy('results/transformer_scaling.json',
            '/kaggle/working/transformer_scaling.json')
print('\nSalvo em /kaggle/working/transformer_scaling.json')

In [ ]:
# ── Célula 7: Resumo dos resultados ────────────────────────────────────────
import json
from pathlib import Path

records = json.loads(Path('results/transformer_scaling.json').read_text())
ok      = [r for r in records if not r.get('skipped')]
oom     = [r for r in records if r.get('oom')]

print(f'Total: {len(records)} entradas | OK: {len(ok)} | OOM: {len(oom)}')
print()
print(f'{"Variant":<26}  {"N":>6}  {"fit":>7}  {"pred":>8}  {"RAM":>8}  {"VRAM":>8}')
print('-' * 72)
for r in records:
    if r.get('skipped'):
        marker = 'OOM' if r.get('oom') else '—'
        print(f"{r['variant']:<26}  {r['n']:>6}  {marker:>7}  {marker:>8}  {marker:>8}  {marker:>8}")
    else:
        print(f"{r['variant']:<26}  {r['n']:>6}  "
              f"{r['fit_s_median']:>6.1f}s  "
              f"{r['pred_ms_median']:>7.1f}ms  "
              f"{r['ram_delta_mb_median']:>6.0f}MB  "
              f"{r['vram_mb_median']:>6.0f}MB")